In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "viktoriia_kalenichenko_silver")
dbutils.widgets.text("silver_table", "environment")

In [0]:
catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
silver_table_name = dbutils.widgets.get("silver_table")

In [0]:
production_silver_table = (f"{catalog}.{silver_schema}.{silver_table_name}")
silver_table = (f"{catalog}.{silver_schema}.{silver_table_name}_schema_lab")

#We copy Silver only during the first launch
if not spark.catalog.tableExists(silver_table):
    spark.sql(f"""
        CREATE TABLE {silver_table}
        DEEP CLONE {production_silver_table}
    """)

print("Experimental table:", silver_table)

##Schema Enforcement


In [0]:
from pyspark.sql import functions as F
silver_df = spark.table(silver_table)

In [0]:
test_df = (
    spark.table(silver_table)
    .limit(1)
    .withColumn("unexpected_column", F.lit("test"))
)

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

try:
    test_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .saveAsTable(silver_table)
except Exception as e:
    print("Write rejected:")
    print(str(e))
else:
    raise AssertionError("Write succeeded — enforcement test failed")

##Schema Evolution

In [0]:
evolution_df = (
    silver_df
    .withColumn(
        "has_missing_metrics",
        F.col("freshwater_per_capita").isNull()
        | F.col("freshwater_withdrawals").isNull()
        | F.col("agricultural_land").isNull()
        | F.col("forest_area").isNull()
    )
)

In [0]:
from delta.tables import DeltaTable

before_count = spark.table(silver_table).count()

# add new column via mergeSchema, but without adding rows
(
    evolution_df.limit(0).write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(silver_table)
)

# fill new columns in existing rows via MERGE.
(
    DeltaTable.forName(spark, silver_table)
    .alias("target")
    .merge(
        evolution_df.alias("source"),
        "target.AREA = source.AREA AND target.YEAR = source.YEAR"
    )
    .whenMatchedUpdate(set={"has_missing_metrics": "source.has_missing_metrics"})
    .execute()
)

after_count = spark.table(silver_table).count()

assert before_count == after_count, "Кількість рядків змінилася!"

print("Rows before:", before_count)
print("Rows after:", after_count)
spark.table(silver_table).printSchema()

In [0]:
#create test column type int
spark.sql(f"""
    ALTER TABLE {silver_table}
    ADD COLUMNS (widening_demo INT)
""")

print("Before widening:")
spark.table(silver_table).select("widening_demo").printSchema()


spark.sql(f"""
    ALTER TABLE {silver_table}
    SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')
""")

# Change INT into BIGINT
spark.sql(f"""
    ALTER TABLE {silver_table}
    ALTER COLUMN widening_demo TYPE BIGINT
""")

print("After widening:")
spark.table(silver_table).select("widening_demo").printSchema()

###Delta column mapping: rename and drop columns safely

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table}
    SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
""")

spark.sql(f"""
    ALTER TABLE {silver_table}
    RENAME COLUMN widening_demo TO widening_demo_renamed
""")

assert "widening_demo_renamed" in spark.table(silver_table).columns
assert "widening_demo" not in spark.table(silver_table).columns

spark.sql(f"""
    ALTER TABLE {silver_table}
    DROP COLUMN widening_demo_renamed
""")

assert "widening_demo_renamed" not in spark.table(silver_table).columns

Data contracts define the expected structure and quality of incoming data before it reaches Silver.
Instead of allowing any schema change silently, a contract can specify required columns, data types, nullable rules, and business constraints. Schema evolution should therefore be controlled and intentional.